
# Semana 2 — SQL avanzado

## Ejercicio 0.1: CTE básica — Estadísticas por zona

**Objetivo:** Utilizar una CTE para calcular estadísticas de los viajes de taxi de Nueva York

**Dataset:** `samples.nyctaxi.trips`

**Análisis:**
- Contar los viajes por zona de origen
- Calcular la tarifa promedio
- Calcular la distancia promedio
- Mostrar únicamente zonas con más de 100 viajes


In [0]:
WITH estadistica_por_zona AS(
    SELECT
        pickup_zip,
        count(*) AS total_viajes,
        AVG(fare_amount) AS promedio_tarifa,
        AVG(trip_distance) AS promedio_distancia
    FROM samples.nyctaxi.trips
    WHERE pickup_zip IS NOT NULL
    GROUP BY pickup_zip
    HAVING total_viajes > 100
    
)

SELECT *
FROM estadistica_por_zona;



# Ejercicio 0.2 — CTE múltiple: Análisis comparativo

## 1. Objetivo

Comparar la tarifa promedio de los viajes de taxi
de Nueva York según la hora de inicio con la tarifa
promedio general del conjunto de datos.

**Dataset:** `samples.nyctaxi.trips`

## 2. Metodología

Se utilizaron dos Common Table Expressions (CTEs).

### CTE 1: promedio_por_hora

Se extrajo la hora de inicio mediante HOUR() y
se agruparon los viajes utilizando GROUP BY.

Para cada hora se calcularon:

- La cantidad total de viajes mediante COUNT(*).
- La tarifa promedio mediante AVG(fare_amount).

### CTE 2: promedio_general

Se utilizó AVG(fare_amount) para calcular la
tarifa promedio de todos los viajes, sin agruparlos.

### Combinación de resultados

Se empleó CROSS JOIN para incorporar el promedio
general a cada uno de los 24 registros horarios.

Finalmente, se calculó la diferencia entre la
tarifa promedio por hora y la tarifa promedio general.

## 3. Resultados

Los resultados obtenidos fueron:

- **Tarifa promedio general:** $12.35.
- **Hora con mayor tarifa promedio:** 05:00.
- **Tarifa promedio máxima por hora:** $14.90.
- **Hora con menor tarifa promedio:** 19:00.
- **Tarifa promedio mínima por hora:** $11.36.
- **Hora con mayor cantidad de viajes:** 18:00,
  con 1,455 viajes.
- **Hora con menor cantidad de viajes:** 05:00,
  con 220 viajes.

## 4. Hallazgos

### Diferencias respecto al promedio general

A las 05:00 se registró una tarifa promedio de
$14.90, superior al promedio general en $2.55.

En cambio, a las 19:00 la tarifa promedio fue
$11.36, inferior al promedio general en $0.99.

Esto demuestra que el promedio de las tarifas
varía según la hora de inicio de los viajes.

### Relación entre cantidad de viajes y tarifas

La mayor cantidad de viajes se registró a las
18:00, con un total de 1,455 viajes y una
tarifa promedio de $11.77.

Por otro lado, a las 05:00 se registró la menor
cantidad de viajes, con 220 registros, pero
también la mayor tarifa promedio: $14.90.

Por tanto, la hora con más viajes no coincide
con la hora de mayor tarifa promedio.

Sin embargo, estos resultados no permiten
establecer una relación causal entre la cantidad
de viajes y el valor de las tarifas.

### Interpretación estadística

La tarifa promedio general fue de $12.35.

Este valor se calculó utilizando todos los
registros individuales, por lo que considera
automáticamente la cantidad de viajes de cada hora.

Al comparar las tarifas horarias con este promedio,
es posible identificar las horas que presentan
valores superiores o inferiores a la referencia.

## 5. Conclusión

El análisis permitió comparar las tarifas promedio
de los viajes según la hora del día utilizando
múltiples CTEs y CROSS JOIN.

Se observó que la tarifa promedio general fue de
$12.35, mientras que el mayor promedio horario
se registró a las 05:00, con $14.90, y el menor
a las 19:00, con $11.36.

Además, se identificó que la hora con mayor
cantidad de viajes (18:00) no presentó la
tarifa promedio más elevada.

Este resultado demuestra que la cantidad de viajes
por hora y la tarifa promedio son indicadores
diferentes y que no deben interpretarse como
si fueran equivalentes.

Desde el punto de vista estadístico, sería
conveniente profundizar en el análisis de las
distancias recorridas, los días de la semana
y los posibles valores atípicos antes de explicar
las causas de estas diferencias.

Desde el punto de vista de Data Engineering,
el ejercicio permitió aplicar CTEs para organizar
consultas complejas y CROSS JOIN para combinar
resultados agregados de diferentes niveles.

## 6. Conceptos aprendidos

- WITH: creación de múltiples CTEs.
- HOUR(): extracción de la hora.
- COUNT(*): conteo de registros.
- AVG(): cálculo de promedios.
- GROUP BY: agrupación de datos.
- CROSS JOIN: combinación de resultados.
- ROUND(): redondeo de valores.
- ORDER BY: ordenamiento de registros.
- AS: asignación de alias.

**Estado:** Ejercicio completado, ejecutado
y documentado con resultados reales.


In [0]:
WITH promedio_por_horas AS(
    SELECT
        
        hour(tpep_pickup_datetime) AS hora,
        COUNT(*) AS total_viajes,
        ROUND(AVG(fare_amount),2) AS tarifa_promedio
        
     FROM samples.nyctaxi.trips
     Group by hour(tpep_pickup_datetime) 
     
),

promedio_general AS (
    SELECT
        ROUND(AVG(fare_amount),2) AS tarifa_promedio_gerenal
    FROM samples.nyctaxi.trips

)

SELECT
    h.hora,
    h.total_viajes,
    h.tarifa_promedio,
    g.tarifa_promedio_gerenal,
    ROUND(h.tarifa_promedio-g.tarifa_promedio_gerenal,2) AS diferencia_tarifa
FROM promedio_por_horas AS h
CROSS JOIN promedio_general AS g

ORDER BY h.hora ASC;







# Ejercicio 0.3 — CTE para limpieza de datos

## 1. Objetivo

Aplicar una Common Table Expression (CTE)
para seleccionar viajes válidos y calcular
estadísticas descriptivas de sus distancias.

**Dataset:** `samples.nyctaxi.trips`

## 2. Metodología

Se creó una CTE llamada `viajes_validos` para
filtrar los registros que cumplen:

- `trip_distance > 0`
- `fare_amount > 0`

Posteriormente, sobre los registros filtrados
se calcularon las siguientes estadísticas:

- MIN(): distancia mínima.
- MAX(): distancia máxima.
- AVG(): distancia promedio.
- MEDIAN(): mediana de las distancias.

Este procedimiento permite separar la etapa
de filtrado de la etapa de análisis estadístico.

## 3. Resultados

Las estadísticas obtenidas fueron:

- **Distancia mínima:** 0.02 millas.
- **Distancia máxima:** 30.60 millas.
- **Distancia promedio:** 2.86 millas.
- **Mediana de distancia:** 1.70 millas.

## 4. Hallazgos

### Comparación entre promedio y mediana

La distancia promedio fue de 2.86 millas,
mientras que la mediana fue de 1.70 millas.

El promedio supera a la mediana en
aproximadamente 1.16 millas.

Esta diferencia sugiere una distribución
asimétrica hacia la derecha, posiblemente
influida por viajes de distancias elevadas.

### Análisis de valores extremos

La distancia mínima registrada fue de
0.02 millas y la máxima de 30.60 millas.

La distancia máxima es considerablemente
superior al promedio y a la mediana.

Por ello, sería recomendable analizar la
distribución mediante percentiles y diagramas
de caja para identificar posibles valores
atípicos.

Sin embargo, un viaje de distancia elevada
no necesariamente constituye un error.

### Importancia de la limpieza de datos

El uso de una CTE permitió excluir registros
con distancias o tarifas no positivas.

No obstante, estas condiciones representan
una validación inicial y no garantizan que
todos los viajes restantes sean correctos.

## 5. Conclusión

El ejercicio permitió aplicar una CTE como
herramienta para realizar un filtrado inicial
de los datos antes del análisis estadístico.

Después de filtrar los registros, se obtuvo
una distancia promedio de 2.86 millas y una
mediana de 1.70 millas.

La diferencia entre ambos indicadores sugiere
que la distribución de las distancias podría
presentar asimetría positiva.

Además, la distancia máxima de 30.60 millas
indica la conveniencia de investigar los
registros extremos para determinar si
corresponden a viajes válidos.

Desde una perspectiva de Data Engineering,
el ejercicio demuestra la importancia de
aplicar reglas de calidad de datos antes
de realizar transformaciones y análisis.

## 6. Conceptos aprendidos

- WITH: creación de una CTE.
- WHERE: filtrado de registros.
- MIN(): cálculo del valor mínimo.
- MAX(): cálculo del valor máximo.
- AVG(): cálculo del promedio.
- MEDIAN(): cálculo de la mediana.
- AS: asignación de alias.
- Limpieza y validación inicial de datos.
- Interpretación de estadísticas descriptivas.

**Estado:** Ejercicio completado, ejecutado
y documentado con resultados reales.


In [0]:
with viajes_validos AS (
    select
        trip_distance,
        fare_amount
    from samples.nyctaxi.trips
    where trip_distance > 0 and fare_amount >0
)

SELECT 
    MIN(trip_distance) AS distancia_minima,
    MAX(trip_distance) AS distancia_maxima,
    AVG(trip_distance) AS distancia_promedio,
    MEDIAN(trip_distance) AS mediana_de_distancia
FROM viajes_validos;
